# Analyse Mt. Kenya data
Follow this procedure to analyse the level2 data

### CAMS data
The part about CAMS data needs to be run only once and downloads the CAMS data to your local computer (taking a long time). 

It requires to have a login at the [Atmosphere data store](https://ads.atmosphere.copernicus.eu/cdsapp#!/home) and an installation of [cdsapi](https://cds.climate.copernicus.eu/api-how-to). 

This data is then read in (from your local computer) and saved as netcdfs in the `./data/level3/cams/` folder. 
From there, the grid cell that is best correlated with the Mt. Kenya measurements is selected. This file is also saved in this git repository (/data/cams/)

If this was already done, this first part can be skipped. 

***If you don't want to download all the CAMS data yourself, you can directly jump to part 4), where the selected CAMS grid for Mt. Kenya are loaded. The cams comparison will still work, some other parts are then not working, however.***




In [1]:
import numpy as np
import os,sys
import xarray as xr
import pandas as pd

import matplotlib.pyplot as plt

%load_ext autoreload
%matplotlib widget

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)

In [2]:
## general settings
stat = "MKN"

dir_data_cams = "../../../Data/CAMS"  # adapt path to local folder to save CAMS data
dir_out="../data/level3/cams" #where to save the processed cams data (in local git repository)

### 1) Get and read CAMS data

In [ ]:
## Get CAMS data from datastore
from analyses.input import get_cams  # require to install cdsapi (https://cds.climate.copernicus.eu/api-how-to)

## Get CAMS data
# This may take several days, and it is only required to be done once
# It was done in Jan 2024, and the data will be saved in seperate netcdf files (see next step)
# TODO wrong path when running here??

# CAMS EGG4 data (CO2, CH4), finally not used:
get_cams.main(
    dir_data=dir_data_cams,
    yr1=2003,
    yr2=2020,
    which="cams_egg4",
    station=[stat],
)
# CAMS EAC4 data (O3, CO, aerosols and more):
get_cams.main(
    dir_data=dir_data_cams,
    yr1=2003,
    yr2=2023,
    which="cams_eac4",
    station=[stat],
)
# CAMS CO2 data:
get_cams.main(
    dir_data=dir_data_cams,
    yr1=2020,
    yr2=2023,
    which="cams_inv_co2",
    station=[stat],
)
# CAMS CH4 data:
get_cams.main(
    dir_data=dir_data_cams,
    yr1=2020,
    yr2=2022,
    which="cams_inv_ch4",
    station=[stat],
)
# CAMS fire data:
get_cams.main(
    dir_data=dir_data_cams,
    yr1=2020,
    yr2=2023,
    which="cams_gfas",
    station=[stat],
) 

In [ ]:
### Save CAMS data as netcdf
import analyses.input.read_cams as read_cams

# Read in the CAMS data and save as netcdfs (one for each cams-dataset)
# The netcdfs will be saved in .\data\level3\cams\

# INVGG CAMS product (those take about a minute):
#co2 invgg with broader resolution (before july 2023):
read_cams.read_cams_inv(dir_data_cams, dir_out=dir_out, species="co2", yr1=2020, yr2=2023,month2=6, station=stat)
#co2 invgg with broader resolution (after july 2023):
read_cams.read_cams_inv(dir_data_cams, dir_out=dir_out, species="co2", yr1=2023, yr2=2023,month1=7, station=stat) 
#ch4 invgg with broader resolution (before july 2022):
read_cams.read_cams_inv(
    dir_data_cams,
    dir_out=dir_out,
    species="ch4",
    yr1=2020,
    yr2=2021,
    dx=3,
    dy=2,
    fact_dxy=2,
    station=stat,
) 
# ch4 invgg finer resolution in 2022
read_cams.read_cams_inv(
    dir_data_cams,
    dir_out=dir_out,
    species="ch4",
    yr1=2022,
    yr2=2023,
    dx=3,
    dy=2,
    fact_dxy=2,
    station=stat,
)  

# Other cams products (those are going faster):
read_cams.read_cams_eac4(dir_data_cams, dir_out=dir_out, station=stat)
read_cams.read_cams_egg4(dir_data_cams, dir_out=dir_out, yr1=2003, yr2=2020, station=stat)
#read_cams.read_cams_gfas(dir_data_cams, dir_out=dir_out) # this takes way too much time!! Better read monthly files individually when needed!

In [6]:
# ## Workaround because CAMS aerosol data was intially not downloaded all at once (should not be needed anymore). 

# import analyses.input.read_cams as read_cams
# from analyses.input import get_cams 

# # The multilevel-mixing-ration data was seperately downloaded with: 
# get_cams.main(
#     dir_data=dir_data_cams + r"\CAMS",
#     yr1=2020,
#     yr2=2023,
#     which="cams_eac4_aerosols",
#     station=[stat],
# )

# # Read in that data that was downloaded and save it as cams_eac4_aerosols_2020_2023_MKN
# read_cams.read_cams_eac4_aerosols(dir_data_cams, dir_out=dir_out, station=stat)

# # Merge this new aerosol data with the previous one:
# # Step 1: Rename the initial NetCDF file
# os.rename(f"{dir_out}/cams_eac4_2003_2023_MKN.nc", f"{dir_out}/cams_eac4_2003_2023_MKN_old.nc")
# # Step2: load the intial (old) netcdf and merge with the new aerosol data.
# aer_old = xr.open_dataset(f"{dir_out}/cams_eac4_2003_2023_MKN_old.nc")
# aer_new = xr.open_dataset(f"{dir_out}/cams_eac4_aerosols_2020_2023_MKN.nc")
# # Step3: merge both datasets and save with the intial filename
# aerosols_all = xr.merge([aer_old,aer_new])
# aerosols_all.to_netcdf(f"{dir_out}/cams_eac4_2003_2023_MKN.nc")

### 2) Read in the gaw kenya data
All the data has to be saved in `../data/` (or another folder given in data_path).\
All data has to be indicated in the dictionary AvailableData (defined in `read_wdc_data.py`). \
If you add new data, please add also an entry to this dictionary (the entry `dataset` has to be a unique name. )

In [ ]:
## Read all data
%autoreload 2
from analyses.input.read_wdc_data import AvailableData, create_data_reader

# File path
data_path = "../data/"

# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
print(all_data)

#####---------- TO ADAPT ---------------#####
selected_data = ['CO2', 'CO2_flask', 
                 'CO', 'CO_flask', 
                 'CH4', 'CH4_flask', 
                 'O3'
                 ] # define data to read in. If empty, all data is used 
## 
processing_kwargs = { 
    'FLASK_FLAG_CORR' : True # exclude flagged flask-data
}
#####-----------------------------------#####

datasets = [] # initialize list of all datasets 
# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path=data_path,dataset=sel,**processing_kwargs) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    # call the data-processing
    data = data_reader.process_data(data)

    # prepare merged dataset
    data = data.drop(columns='endtime') # problem when merging datasets (because of NaT?), so better remove endtime
    ds = data.to_xarray()
    ds = ds.assign_coords(dataset=sel)
    ds['species'] = data_reader.species
    ds['unit']  = np.unique(ds.unit.dropna(dim='time'))[0]
    datasets.append(ds)

# save all in one xarray dataset
ds_all = xr.concat(datasets,dim="dataset")


### 3) Compare to CAMS
Select for each dataset the best-correlated cams grid at the MKN station. 
Save that cams-data as a seperate netcdf


In [ ]:
from analyses.input import read_cams 

In [ ]:
# Get the best CAMS for each dataset, by checking the correlation to the measurement data
## this takes some time (need to do it once, then just load the nc file in the next cell)
cams_best_grid = read_cams.get_best_cams(ds_all, dir_out=dir_out, save_netcdf=True)



In [ ]:
# once it is saved, it can be loaded with: 
cams_best_grid = xr.open_dataset(f"{data_path}/level3/cams/cams_best_grid_MKN.nc")

#### Process cams_best_grid

The CAMS inversion GHG flux data has a different horizontal resolution for newer data.
The best grid was therefore selected seperately for the 2 periods (before and after resolution change) and saved in the dataset as 'co2_invgg' and 'co2_invgg2' (and the same for ch4). 
The selected grids can thus be checked for both periods. 

For easier handling, we merge those seperate datasets in the following and save as 'co2_invgg', and save it as a seperate dataset cams_best_grid_merged_MKN.nc



In [ ]:
cams_best_grid_merged = cams_best_grid.copy()

for s in ["co2", "ch4"]:
    # select the two datasets to merge
    cams_sel1 = cams_best_grid.sel(dataset=f"{s}_invgg")
    cams_sel2 = cams_best_grid.sel(dataset=f"{s}_invgg2")
    # merge the two datasets, using the values of the first dataset where they are not nan, otherwise use the second dataset (so here it should just add the most recent data)
    merged_values = cams_sel1.combine_first(cams_sel2) # merge the two datasets, using the values of the first dataset where they are not nan
    # replace the initial dataset with the merged dataset
    cams_best_grid_merged['value'].loc[dict(dataset=f"{s}_invgg")] = merged_values.value
    # remove the second initial dataset that was now merged
    cams_best_grid_merged = cams_best_grid_merged.drop_sel(dataset=f"{s}_invgg2")

cams_best_grid_merged.to_netcdf(f"{data_path}/level3/cams/cams_best_grid_merged_MKN.nc")

### 4) Compare Mt. Kenya data with CAMS

In [ ]:
# Load the selected CAMS data
cams_best_grid = xr.open_dataset(f"{data_path}/level3/cams/cams_best_grid_merged_MKN.nc")

# Load the desired ground-based data: 
# run the cell (2) above
ds_all